# Covariate, Target Neuron and Prediction Rate Maps — M29 D23

Two target cells are analysed: a **grid cell** (unit 214) and an **NGS cell** (unit 340).

For each target, XGBoost is fitted with:
- Behavioural baselines (null / pos / pos+spd+lfp)
- **Random-10**: 10 randomly chosen GC cells and 10 NGS cells
- **N-equal**: equal numbers of GC and NGS cells — using all cells of the rarer type
  and a matched random sample of the more abundant type

The covariate trace panel always shows 10 GC and 10 NGS cells.
Predicted spike traces have Poisson-sampled spikes overlaid as tick marks.

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from spatial_manifolds.detect_grids import *
from spatial_manifolds.mlencoding import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

plt.rcParams['font.family'] = 'Arial'

mouse       = 29
day         = 23
source_path = '/Users/harryclark/Downloads/COHORT12/'
savepath_vr  = (f'/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
                f'figure_4_xgboost/M{mouse}D{day}_rate_map_cov_traces_VR.pdf')
savepath_of  = (f'/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
                f'figure_4_xgboost/M{mouse}D{day}_rate_map_cov_traces_OF1.pdf')
savepath_combined = (f'/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
                     f'figure_4_xgboost/M{mouse}D{day}_rate_map_cov_traces_combined.pdf')

# ── Target cells ──────────────────────────────────────────────────────────────
test_grid_cell_1 = 214   # co-modular grid cell
test_ngs_cell_1  = 340   # non-grid spatial cell
TARGET_CELLS = [test_grid_cell_1, test_ngs_cell_1]

# ── Model parameters ─────────────────────────────────────────────────────────
HISTORY_LENGTH  = 1000
N_CV            = 5
N_RANDOM_COV    = 10      # cells per type in the random-10 condition
N_DISPLAY_COV   = 10      # cells per type shown in covariate trace panel
TRACE_WINDOW_VR = (24150, 26250)
TRACE_GAIN_VR   = 5

N_BINS_2D = 30
SIGMA_2D  = 1.5

COL_GC  = '#c04744'
COL_NGS = '#3171ae'
TARGET_COLOR = {test_grid_cell_1: COL_GC, test_ngs_cell_1: COL_NGS}
TARGET_LABEL = {test_grid_cell_1: f'GC (unit {test_grid_cell_1})',
                test_ngs_cell_1:  f'NGS (unit {test_ngs_cell_1})'}

def white_to_hex_cmap(hex_color):
    return LinearSegmentedColormap.from_list('cmap', ['#FFFFFF', hex_color])

rng = np.random.default_rng(42)

cell_class = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
sess_cells = cell_class[
    (cell_class['mouse'] == mouse) & (cell_class['day'] == day)
].copy()
sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
print(f'M{mouse} D{day}: {len(sess_cells)} cells')
print(f'Target cells: GC={test_grid_cell_1}  NGS={test_ngs_cell_1}')

## 1. Load session data

In [ ]:
print('Loading VR...')
tcs_vr, tcs_time_vr, _, last_ephys_bin_vr, beh_vr, clusters_vr = compute_vr_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False, source_path=source_path)
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')

print('Loading OF1...')
tcs_of, tcs_time_of, beh_of, clusters_of, ep_of = compute_of_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path, session='OF1')

# ── Classify cells ────────────────────────────────────────────────────────────
gcs, ngs, all_cells = classify_cells_both_sessions(
    mouse, day, source_path=source_path)
gc_ids  = gcs.cluster_id.values.astype(int)
ngs_ids = ngs.cluster_id.values.astype(int)
print(f'VR: {len(tcs_time_vr)} cells  |  OF1: {len(tcs_time_of)} cells')
print(f'GC: {len(gc_ids)}  NGS: {len(ngs_ids)}')

In [ ]:
def get_vr_signals(beh, ep, T):
    def _b(k):
        a = np.array(beh[k].bin_average(bin_size=time_bs, time_units='ms', ep=ep))
        return pd.Series(a).ffill().bfill().values[:T]
    dt  = _b('travel') - ((beh['trial_number'][0] - 1) * tl)
    return dt % tl, _b('S')

def get_of_signals(beh, ep, T):
    def _b(k):
        a = np.array(beh[k].bin_average(bin_size=time_bs, time_units='ms', ep=ep))
        return pd.Series(a).ffill().bfill().values[:T]
    return _b('head_x'), _b('head_y'), _b('S'), _b('H'), _b('Hing')

def _pad(arr, T):
    arr = np.array(arr)[:T]
    return np.pad(arr, (0, max(0, T - len(arr))))

def make_of_rate_map(spikes, px, py, n_bins=N_BINS_2D, sigma=SIGMA_2D):
    T = min(len(spikes), len(px), len(py))
    xe = np.linspace(np.nanpercentile(px[:T],1), np.nanpercentile(px[:T],99), n_bins+1)
    ye = np.linspace(np.nanpercentile(py[:T],1), np.nanpercentile(py[:T],99), n_bins+1)
    tc,  _,_ = np.histogram2d(px[:T], py[:T], bins=[xe,ye], weights=spikes[:T])
    occ,_,_ = np.histogram2d(px[:T], py[:T], bins=[xe,ye])
    return gaussian_filter(np.nan_to_num(tc/np.where(occ>0,occ,np.nan)).T, sigma=sigma)

print('Helpers defined.')

## 2. Select covariate cells and fit VR XGBoost models

Both target cells (GC 214 and NGS 340) are excluded from all covariate pools.  
GC and NGS pools are then balanced to produce two conditions:

- **Random-10**: 10 randomly chosen GC + 10 NGS cells  
- **N-equal**: all cells of the rarer type + a matched sample of the more abundant type

The covariate trace panel uses a separate display sample of 10 GC + 10 NGS cells.

In [ ]:
nfilters = int(HISTORY_LENGTH / time_bs)
xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                  window=time_bs, n_filters=nfilters, max_time=HISTORY_LENGTH)

# ── Covariate pools ───────────────────────────────────────────────────────────
gcs, ngs, all_cells = classify_cells_both_sessions(mouse, day, source_path=source_path)
gc_ids  = set(gcs.cluster_id.values.astype(int))
ngs_ids = set(ngs.cluster_id.values.astype(int))
exclude = set(TARGET_CELLS)

pool_gc  = sorted([c for c in gc_ids  if c in tcs_time_vr and c not in exclude])
pool_ngs = sorted([c for c in ngs_ids if c in tcs_time_vr and c not in exclude])
N_equal  = min(len(pool_gc), len(pool_ngs))
print(f'GC pool: {len(pool_gc)}  NGS pool: {len(pool_ngs)}  N_equal: {N_equal}')

rand_gc  = list(rng.choice(pool_gc,  size=min(N_RANDOM_COV, len(pool_gc)),  replace=False).astype(int))
rand_ngs = list(rng.choice(pool_ngs, size=min(N_RANDOM_COV, len(pool_ngs)), replace=False).astype(int))
if len(pool_gc) <= len(pool_ngs):
    equal_gc  = pool_gc[:]
    equal_ngs = list(rng.choice(pool_ngs, size=N_equal, replace=False).astype(int))
else:
    equal_ngs = pool_ngs[:]
    equal_gc  = list(rng.choice(pool_gc, size=N_equal, replace=False).astype(int))
disp_gc  = list(rng.choice(pool_gc,  size=min(N_DISPLAY_COV, len(pool_gc)),  replace=False).astype(int))
disp_ngs = list(rng.choice(pool_ngs, size=min(N_DISPLAY_COV, len(pool_ngs)), replace=False).astype(int))
print(f'Rand-10: {len(rand_gc)} GC + {len(rand_ngs)} NGS  |  N-equal: {len(equal_gc)} GC + {len(equal_ngs)} NGS')

# ── VR behavioral signals ─────────────────────────────────────────────────────
T_vr = min(len(np.array(tcs_time_vr[tid])) for tid in TARGET_CELLS)
pos_vr, spd_vr = get_vr_signals(beh_vr, ep_vr, T_vr)

def make_cov_mat(ids, T):
    return np.vstack([_pad(np.array(tcs_time_vr[nid]), T) for nid in ids]).T

cov_mats = {
    'rand_gc':   make_cov_mat(rand_gc,  T_vr),
    'rand_ngs':  make_cov_mat(rand_ngs, T_vr),
    'equal_gc':  make_cov_mat(equal_gc, T_vr),
    'equal_ngs': make_cov_mat(equal_ngs, T_vr),
    'equal_both': np.hstack([make_cov_mat(equal_gc, T_vr), make_cov_mat(equal_ngs, T_vr)]),
}

# ── Model spec factory ────────────────────────────────────────────────────────
# Returns list of (label, description, x_b) for a given target cell
def make_vr_models(tid, T, pos, spd, lfp):
    null  = np.zeros((T, 1))
    p     = pos[:T, None]
    ps    = np.column_stack([pos[:T], spd[:T]])
    psl   = np.column_stack([pos[:T], spd[:T], lfp])
    specs = []
    # ── Baselines ─────────────────────────────────────────────────────────────
    for bl_label, bl_desc, x_bl in [
        ('null',       'Null',       null),
        ('pos',        'Pos',        p),
        ('pos+spd',    'P+S',        ps),
        ('pos+spd+lfp','P+S+LFP',   psl),
    ]:
        specs.append((bl_label, bl_desc, x_bl))
        # ── + each cell group ─────────────────────────────────────────────────
        for cov_key, cov_desc, n_cells in [
            ('rand_gc',   f'+{len(rand_gc)} GC (rand)',   len(rand_gc)),
            ('rand_ngs',  f'+{len(rand_ngs)} NGS (rand)', len(rand_ngs)),
            ('equal_gc',  f'+{N_equal} GC (equal)',       N_equal),
            ('equal_ngs', f'+{N_equal} NGS (equal)',      N_equal),
        ]:
            cm = cov_mats[cov_key][:T]
            specs.append(
                (f'{bl_label}+{cov_key}',
                 f'{bl_desc} {cov_desc}',
                 np.column_stack([x_bl, cm]) if x_bl.shape[1] > 0 else cm)
            )
    # ── Final: full baseline + both N-equal groups ────────────────────────────
    specs.append((
        'psl+equal_both',
        f'P+S+LFP +{N_equal} GC +{N_equal} NGS',
        np.column_stack([psl, cov_mats['equal_both'][:T]])
    ))
    return specs

# ── Fit all models for each VR target ────────────────────────────────────────
models_vr   = {}
rm_preds_vr = {}
rm_true_vr  = {}

for tid in TARGET_CELLS:
    y = np.array(tcs_time_vr[tid])
    T = len(y)
    print(f'\n── VR target {tid} ({TARGET_LABEL[tid]}) ──')
    try:
        lfp = _pad(get_theta_trace(mouse=mouse, day=day, cluster_id=tid,
                                   time_bs=50, resample_bs=time_bs,
                                   session_type='VR', source_path=source_path), T)
        print('  LFP loaded')
    except Exception as e:
        print(f'  LFP fallback: {e}'); lfp = np.zeros(T)

    specs = make_vr_models(tid, T, pos_vr, spd_vr, lfp)
    models_vr[tid] = {}
    for label, desc, x_b in specs:
        Y_hat, pr2 = xgb.fit_cv(x_b, y, verbose=0, continuous_folds=True, n_cv=N_CV)
        models_vr[tid][label] = {'Y_hat': Y_hat, 'pR2': float(np.nanmean(pr2)), 'desc': desc}
        print(f'  [{label:30s}] {str(x_b.shape):14s}  pR²={models_vr[tid][label]["pR2"]:+.4f}')

    # Reconstruct rate maps for curated subset only (to limit reconstruction calls)
    RECON_KEYS = [
        'null', 'pos', 'pos+spd+lfp',
        'pos+rand_gc', 'pos+rand_ngs',
        'pos+equal_gc', 'pos+equal_ngs',
        'pos+spd+lfp+rand_gc', 'pos+spd+lfp+rand_ngs',
        'pos+spd+lfp+equal_gc', 'pos+spd+lfp+equal_ngs',
        'psl+equal_both',
    ]
    _excl   = set(TARGET_CELLS) | set(rand_gc) | set(rand_ngs) | set(equal_gc) | set(equal_ngs)
    _pseudo = [c for c in all_cells.cluster_id.values.astype(int) if c not in _excl]
    exp_spk = {_pseudo[j]: models_vr[tid][lbl]['Y_hat']
               for j, lbl in enumerate(RECON_KEYS)}
    tcs_r, _, _, lb_r, _, _ = compute_vr_tcs_using_expected_spikes(
        mouse, day, apply_zscore=False, vr_type='VR',
        source_path=source_path, expected_spikes=exp_spk)
    s = 2.5
    rm_true_vr[tid] = gaussian_filter(
        np.nan_to_num(tcs_vr[tid]).astype(np.float64), sigma=s)[:lb_r]
    rm_preds_vr[tid] = {
        lbl: gaussian_filter(np.nan_to_num(tcs_r[_pseudo[j]]).astype(np.float64), sigma=s)[:lb_r]
        for j, lbl in enumerate(RECON_KEYS)
    }

print('\nVR fitting done.')

## 3. VR rate map figure

In [ ]:
# Curated display: baselines | random-10 group | N-equal group | final
SHOW_VR_GROUPS = [
    # (label, col_title, group_name)
    ('null',                    'Null',                    'baseline'),
    ('pos',                     'Pos',                     'baseline'),
    ('pos+spd+lfp',             'P+S+LFP',                 'baseline'),
    ('pos+rand_gc',             f'P+{len(rand_gc)}GC(r)',  'random'),
    ('pos+rand_ngs',            f'P+{len(rand_ngs)}NGS(r)','random'),
    ('pos+spd+lfp+rand_gc',     f'PSL+{len(rand_gc)}GC(r)','random'),
    ('pos+spd+lfp+rand_ngs',    f'PSL+{len(rand_ngs)}NGS(r)','random'),
    ('pos+equal_gc',            f'P+{N_equal}GC(eq)',      'equal'),
    ('pos+equal_ngs',           f'P+{N_equal}NGS(eq)',     'equal'),
    ('pos+spd+lfp+equal_gc',    f'PSL+{N_equal}GC(eq)',    'equal'),
    ('pos+spd+lfp+equal_ngs',   f'PSL+{N_equal}NGS(eq)',   'equal'),
    ('psl+equal_both',          f'PSL+{N_equal}GC+{N_equal}NGS', 'final'),
]

GROUP_SEP = {'baseline': '#dddddd', 'random': '#ffd7b5', 'equal': '#b5d7ff', 'final': '#c8e6c9'}
n_show = len(SHOW_VR_GROUPS); n_rows = len(TARGET_CELLS)

fig_vr, axes = plt.subplots(n_rows, n_show+1,
                             figsize=((n_show+1)*1.35, n_rows*3.0),
                             gridspec_kw={'hspace':0.18, 'wspace':0.06})
if n_rows == 1: axes = axes[np.newaxis,:]

for ri, tid in enumerate(TARGET_CELLS):
    color = TARGET_COLOR[tid]; is_top = (ri==0)
    ax = axes[ri, 0]
    plot_firing_rate_map(ax, rm_true_vr[tid], bs=bs, tl=tl, p=95, cmap=white_to_hex_cmap(color))
    ax.set_ylabel(TARGET_LABEL[tid], fontsize=8, labelpad=3)
    if is_top: ax.set_title('True', fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=6)

    for pi, (key, title, grp) in enumerate(SHOW_VR_GROUPS):
        ax = axes[ri, pi+1]
        plot_firing_rate_map(ax, rm_preds_vr[tid][key], bs=bs, tl=tl, p=95, cmap=white_to_hex_cmap(color))
        ax.set_facecolor(GROUP_SEP[grp])
        pr2 = models_vr[tid][key]['pR2']
        if is_top:
            ax.set_title(f'{title}\n{pr2:+.3f}', fontsize=5.5, color=color, pad=2)
        else:
            ax.set_title(f'{pr2:+.3f}', fontsize=5.5, color=color, pad=2)
        ax.set_yticks([]); ax.tick_params(labelsize=5)

# Group header legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=c, label=g, edgecolor='#999')
              for g,c in GROUP_SEP.items()]
fig_vr.legend(handles=legend_els, ncol=4, fontsize=7, loc='lower center',
              bbox_to_anchor=(0.5, -0.03), frameon=False)
fig_vr.suptitle(f'VR — M{mouse} D{day}  |  N_equal={N_equal}', fontsize=10, fontweight='bold')
fig_vr.savefig(savepath_vr, bbox_inches='tight', dpi=200)
plt.show(); print(f'Saved → {savepath_vr}')

In [ ]:
# ── pR² summary: all models for both targets ──────────────────────────────────
all_model_keys = list(next(iter(models_vr.values())).keys())
fig_pr2, axes_pr2 = plt.subplots(1, len(TARGET_CELLS),
                                   figsize=(max(len(all_model_keys)*0.35, 8), 4),
                                   sharey=False)
if len(TARGET_CELLS) == 1: axes_pr2 = [axes_pr2]

for ax, tid in zip(axes_pr2, TARGET_CELLS):
    color  = TARGET_COLOR[tid]
    keys   = all_model_keys
    pr2s   = [models_vr[tid][k]['pR2'] for k in keys]
    descs  = [models_vr[tid][k]['desc'] for k in keys]
    x      = np.arange(len(keys))
    bars   = ax.bar(x, pr2s, color=color, alpha=0.7, edgecolor='#555555', linewidth=0.5)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(x)
    ax.set_xticklabels(descs, rotation=60, ha='right', fontsize=5.5)
    ax.set_ylabel('pR²', fontsize=9)
    ax.set_title(TARGET_LABEL[tid], fontsize=9, color=color, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    # Group shading
    grp_labels = [v[2] for v in make_vr_models(tid, T_vr, pos_vr, spd_vr, np.zeros(T_vr))]
    grp_labels.append('final')  # psl+equal_both
    grp_x = {}  # track x positions per group
    for xi, k in enumerate(keys):
        grp = next((v[2] for v in make_vr_models(tid, T_vr, pos_vr, spd_vr, np.zeros(T_vr)) if v[0]==k), 'final')
        grp_x.setdefault(grp, []).append(xi)
    for grp, xs in grp_x.items():
        ax.axvspan(min(xs)-0.5, max(xs)+0.5, alpha=0.07,
                   color={'baseline':'grey','random':'orange','equal':'steelblue','final':'green'}.get(grp,'grey'))

fig_pr2.suptitle(f'VR pR² — all models — M{mouse} D{day}', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(savepath_vr.replace('.pdf','_pr2_summary.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 4. VR — covariate and prediction traces

In [ ]:
SHOW_TRACES_VR = [
    ('null',           'Null'),
    ('pos',            'Pos'),
    ('pos+spd+lfp',    'P+S+LFP'),
    ('pos+rand10_gc',  f'P+{len(rand_gc)} GC (rand)'),
    ('pos+rand10_ngs', f'P+{len(rand_ngs)} NGS (rand)'),
    ('pos+equal_gc',   f'P+{len(equal_gc)} GC (equal)'),
    ('pos+equal_ngs',  f'P+{len(equal_ngs)} NGS (equal)'),
]

def _norm01(arr, win):
    seg = np.array(arr)[win].astype(float)
    lo, hi = np.nanmin(seg), np.nanmax(seg)
    return (seg - lo) / (hi - lo + 1e-10)

def draw_covariates(ax, pos, spd, gc_tr, ngs_tr, window):
    win   = slice(*window); t_idx = np.arange(*window)
    h = 1.2; lpad = len(t_idx)*0.30; wf = len(t_idx)*0.02
    traces = (
        [(pos,'Pos','black',1.0,1.3),(spd,'Speed','#9b59b6',1.0,1.3)] +
        [(t,f'GC{i+1}',  COL_GC,  0.75,0.9) for i,t in enumerate(gc_tr)] +
        [(t,f'NGS{i+1}', COL_NGS, 0.75,0.9) for i,t in enumerate(ngs_tr)]
    )
    n = len(traces)
    for j,(arr,lbl,c,a,lw) in enumerate(traces):
        off = h*(n-1-j)
        ax.plot(t_idx, _norm01(arr,win)+off, lw=lw, alpha=a, color=c)
        ax.text(t_idx[0]-wf, off+h*0.05, lbl, ha='right', va='bottom', fontsize=4.5, color=c)
    for div in [2, 2+len(gc_tr)]:
        ax.axhline(h*(n-div)-h*0.1, color='lightgray', lw=0.5, ls='--')
    bar_len = int(1000/time_bs); bar_y = -h*0.7
    ax.plot([t_idx[0], t_idx[0]+bar_len], [bar_y,bar_y], 'k-', lw=2)
    ax.text(t_idx[0]+bar_len*0.5, bar_y-h*0.15, '1 s', ha='center', va='top', fontsize=5.5)
    ax.set_xlim(t_idx[0]-lpad, t_idx[-1]+len(t_idx)*0.03)
    ax.set_ylim(bar_y-h*0.4, h*n+0.6); ax.axis('off')

def draw_preds(ax, y_true, models_d, show_list, window, gain, color, seed=0):
    rng_spk = np.random.default_rng(seed)
    win = slice(*window); t_idx = np.arange(*window); wf = len(t_idx)*0.02
    n   = len(show_list)
    true = np.array(y_true)[win]
    peak = float(np.nanmax(np.abs(true))); step = peak*1.5 if peak>0 else 1.0
    tick_h = step*0.18; top = step*n
    ax.plot(t_idx, true*2+top, lw=1.2, color=color, alpha=0.5)
    for b in np.where(true>0)[0]:
        yb = true[b]*2+top; ax.plot([t_idx[b],t_idx[b]],[yb,yb+tick_h], color=color, lw=0.9, alpha=0.9)
    ax.text(t_idx[0]-wf, top+step*0.05, 'True', ha='right', va='bottom', fontsize=5.5, color=color)
    for j,(key,lbl) in enumerate(show_list):
        off  = step*(n-1-j)
        pred = np.maximum(models_d[key]['Y_hat'][win], 0)
        pr2  = models_d[key]['pR2']
        ax.plot(t_idx, pred*gain+off, lw=1.0, alpha=0.55, color=color)
        for b,cnt in enumerate(rng_spk.poisson(pred)):
            for _ in range(cnt):
                yb = pred[b]*gain+off; ax.plot([t_idx[b],t_idx[b]],[yb,yb+tick_h], color=color, lw=0.8, alpha=0.85)
        ax.text(t_idx[0]-wf, off+step*0.05, lbl, ha='right', va='bottom', fontsize=4.5, color=color)
        ax.text(t_idx[-1], off+step*0.05, f'{pr2:.2f}', ha='left', va='bottom', fontsize=4.5, color='dimgray')
    ax.set_xlim(t_idx[0]-len(t_idx)*0.30, t_idx[-1]+len(t_idx)*0.14)
    ax.set_ylim(-step*0.3, top+step*1.1); ax.axis('off')

# ── Build figure: one row per target ─────────────────────────────────────────
n_show  = len(SHOW_TRACES_VR)
n_rows  = len(TARGET_CELLS)
col_w   = [2.0, 0.1, 2.0, 0.2, 1.0, 0.1] + [1.0]*n_show
n_cols  = len(col_w)

gc_display_traces  = [np.array(tcs_time_vr[c]) for c in disp_gc]
ngs_display_traces = [np.array(tcs_time_vr[c]) for c in disp_ngs]

fig_vt = plt.figure(figsize=(sum(col_w)*0.78, n_rows*3.2))
gs_vt  = gridspec.GridSpec(n_rows, n_cols, figure=fig_vt,
                            width_ratios=col_w, hspace=0.18, wspace=0.10)

for ri, tid in enumerate(TARGET_CELLS):
    color  = TARGET_COLOR[tid]; is_top = (ri==0)
    y_true = np.array(tcs_time_vr[tid])

    draw_covariates(fig_vt.add_subplot(gs_vt[ri, 0]),
                    pos_vr, spd_vr, gc_display_traces, ngs_display_traces, TRACE_WINDOW_VR)
    if is_top: fig_vt.axes[-1].set_title(f'Covariates ({N_DISPLAY_COV} GC + {N_DISPLAY_COV} NGS)', fontsize=7.5, pad=4)
    fig_vt.axes[-1].text(-0.02, 0.5, TARGET_LABEL[tid], transform=fig_vt.axes[-1].transAxes,
                         fontsize=7, color=color, fontweight='bold', va='center', ha='right', rotation=90)

    fig_vt.add_subplot(gs_vt[ri, 1]).axis('off')

    draw_preds(fig_vt.add_subplot(gs_vt[ri, 2]),
               y_true, models_vr[tid], SHOW_TRACES_VR, TRACE_WINDOW_VR, TRACE_GAIN_VR, color, seed=ri)
    if is_top: fig_vt.axes[-1].set_title('Predicted traces (Poisson spikes)', fontsize=7.5, pad=4)

    fig_vt.add_subplot(gs_vt[ri, 3]).axis('off')

    ax = fig_vt.add_subplot(gs_vt[ri, 4])
    plot_firing_rate_map(ax, rm_true_vr[tid], bs=bs, tl=tl, p=95, cmap=white_to_hex_cmap(color))
    if is_top: ax.set_title('True', fontsize=7.5, fontweight='bold')
    ax.set_ylabel('Trial', fontsize=6); ax.tick_params(labelsize=6)

    fig_vt.add_subplot(gs_vt[ri, 5]).axis('off')

    for pi, (key, lbl) in enumerate(SHOW_TRACES_VR):
        ax = fig_vt.add_subplot(gs_vt[ri, pi+6])
        plot_firing_rate_map(ax, rm_preds_vr[tid][key], bs=bs, tl=tl, p=95, cmap=white_to_hex_cmap(color))
        if is_top: ax.set_title(f'{lbl}\npR²={models_vr[tid][key]["pR2"]:+.3f}', fontsize=6, color=color)
        else:      ax.set_title(f'pR²={models_vr[tid][key]["pR2"]:+.3f}', fontsize=6, color=color)
        ax.set_yticks([]); ax.tick_params(labelsize=6)

fig_vt.suptitle(f'VR — M{mouse} D{day}  (random-10 vs N-equal={N_equal})',
                fontsize=9, fontweight='bold', y=1.01)
fig_vt.savefig(savepath_vr, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {savepath_vr}')

## 5. OF1 — fit XGBoost models and build rate maps
Same covariate sets as VR, applied to the OF1 session.
Predicted rate maps are 2D and reconstructed via `compute_of_tcs_using_expected_spikes`.

In [ ]:
T_of = min(len(np.array(tcs_time_of[tid])) for tid in TARGET_CELLS)
px_of, py_of, spd_of, hd_of, hing_of = get_of_signals(beh_of, ep_of, T_of)

def make_cov_mat_of(ids, T):
    return np.vstack([_pad(np.array(tcs_time_of[nid]), T) for nid in ids]).T

cov_mats_of = {
    'rand_gc':    make_cov_mat_of(rand_gc,  T_of),
    'rand_ngs':   make_cov_mat_of(rand_ngs, T_of),
    'equal_gc':   make_cov_mat_of(equal_gc, T_of),
    'equal_ngs':  make_cov_mat_of(equal_ngs, T_of),
    'equal_both': np.hstack([make_cov_mat_of(equal_gc, T_of), make_cov_mat_of(equal_ngs, T_of)]),
}

def make_of_models(tid, T, px, py, spd, hd, hing, lfp):
    null  = np.zeros((T, 1))
    p     = np.column_stack([px[:T], py[:T]])
    ps    = np.column_stack([px[:T], py[:T], spd[:T]])
    full  = np.column_stack([px[:T], py[:T], spd[:T], hd[:T], hing[:T], lfp])
    specs = []
    for bl_label, bl_desc, x_bl in [
        ('null',    'Null',    null),
        ('pos',     'Pos',     p),
        ('pos+spd', 'P+S',     ps),
        ('full_beh','Full beh',full),
    ]:
        specs.append((bl_label, bl_desc, x_bl))
        for cov_key, cov_desc in [
            ('rand_gc',  f'+{len(rand_gc)} GC (rand)'),
            ('rand_ngs', f'+{len(rand_ngs)} NGS (rand)'),
            ('equal_gc', f'+{N_equal} GC (equal)'),
            ('equal_ngs',f'+{N_equal} NGS (equal)'),
        ]:
            cm = cov_mats_of[cov_key][:T]
            specs.append((f'{bl_label}+{cov_key}', f'{bl_desc} {cov_desc}',
                          np.column_stack([x_bl, cm])))
    specs.append((
        'full+equal_both',
        f'Full +{N_equal} GC +{N_equal} NGS',
        np.column_stack([full, cov_mats_of['equal_both'][:T]])
    ))
    return specs

models_of   = {}
rm_preds_of = {}
rm_true_of  = {}

for tid in TARGET_CELLS:
    y = np.array(tcs_time_of[tid])
    T = len(y)
    print(f'\n── OF1 target {tid} ({TARGET_LABEL[tid]}) ──')
    try:
        lfp = _pad(get_theta_trace(mouse=mouse, day=day, cluster_id=tid,
                                   time_bs=50, resample_bs=time_bs,
                                   session_type='OF1', source_path=source_path), T)
        print('  LFP loaded')
    except Exception as e:
        print(f'  LFP fallback: {e}'); lfp = np.zeros(T)

    specs = make_of_models(tid, T, px_of, py_of, spd_of, hd_of, hing_of, lfp)
    models_of[tid] = {}
    for label, desc, x_b in specs:
        Y_hat, pr2 = xgb.fit_cv(x_b, y, verbose=0, continuous_folds=True, n_cv=N_CV)
        models_of[tid][label] = {'Y_hat': Y_hat, 'pR2': float(np.nanmean(pr2)), 'desc': desc}
        print(f'  [{label:30s}] {str(x_b.shape):14s}  pR²={models_of[tid][label]["pR2"]:+.4f}')

    RECON_KEYS_OF = [
        'null', 'pos', 'full_beh',
        'pos+rand_gc', 'pos+rand_ngs',
        'pos+equal_gc', 'pos+equal_ngs',
        'full_beh+rand_gc', 'full_beh+rand_ngs',
        'full_beh+equal_gc', 'full_beh+equal_ngs',
        'full+equal_both',
    ]
    _excl   = set(TARGET_CELLS) | set(rand_gc) | set(rand_ngs) | set(equal_gc) | set(equal_ngs)
    _pseudo = [c for c in all_cells.cluster_id.values.astype(int) if c not in _excl]
    exp_spk = {_pseudo[j]: models_of[tid][lbl]['Y_hat']
               for j, lbl in enumerate(RECON_KEYS_OF)}
    tcs_r_of, _, _, _, _ = compute_of_tcs_using_expected_spikes(
        mouse, day, apply_zscore=False, apply_guassian_filter=True,
        source_path=source_path, expected_spikes=exp_spk)
    rm_true_of[tid] = gaussian_filter(
        np.nan_to_num(tcs_of[tid]).astype(np.float64), sigma=SIGMA_2D)
    rm_preds_of[tid] = {
        lbl: np.nan_to_num(tcs_r_of[_pseudo[j]]).astype(np.float64)
        for j, lbl in enumerate(RECON_KEYS_OF)
    }

print('\nOF1 fitting done.')

## 6. OF1 rate map figure

In [ ]:
SHOW_OF_GROUPS = [
    ('null',              'Null',                   'baseline'),
    ('pos',               'Pos',                    'baseline'),
    ('full_beh',          'Full beh',               'baseline'),
    ('pos+rand_gc',       f'P+{len(rand_gc)}GC(r)', 'random'),
    ('pos+rand_ngs',      f'P+{len(rand_ngs)}NGS(r)','random'),
    ('full_beh+rand_gc',  f'Full+{len(rand_gc)}GC(r)','random'),
    ('full_beh+rand_ngs', f'Full+{len(rand_ngs)}NGS(r)','random'),
    ('pos+equal_gc',      f'P+{N_equal}GC(eq)',     'equal'),
    ('pos+equal_ngs',     f'P+{N_equal}NGS(eq)',    'equal'),
    ('full_beh+equal_gc', f'Full+{N_equal}GC(eq)',  'equal'),
    ('full_beh+equal_ngs',f'Full+{N_equal}NGS(eq)', 'equal'),
    ('full+equal_both',   f'Full+{N_equal}GC+{N_equal}NGS','final'),
]

n_show = len(SHOW_OF_GROUPS); n_rows = len(TARGET_CELLS)
fig_of, axes = plt.subplots(n_rows, n_show+1,
                             figsize=((n_show+1)*2.0, n_rows*2.8),
                             gridspec_kw={'hspace':0.15,'wspace':0.08})
if n_rows == 1: axes = axes[np.newaxis,:]

GROUP_SEP = {'baseline':'#dddddd','random':'#ffd7b5','equal':'#b5d7ff','final':'#c8e6c9'}

for ri, tid in enumerate(TARGET_CELLS):
    color = TARGET_COLOR[tid]; is_top = (ri==0)
    vmax = max(rm_true_of[tid].max(),
               max(rm_preds_of[tid][k].max() for k,_,_ in SHOW_OF_GROUPS))
    ax = axes[ri,0]
    ax.imshow(rm_true_of[tid], origin='lower', cmap='viridis', vmin=0, vmax=vmax, interpolation='nearest')
    ax.set_ylabel(TARGET_LABEL[tid], fontsize=8)
    if is_top: ax.set_title('True', fontsize=8, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    for pi,(key,title,grp) in enumerate(SHOW_OF_GROUPS):
        ax = axes[ri, pi+1]
        ax.imshow(rm_preds_of[tid][key], origin='lower', cmap='viridis',
                  vmin=0, vmax=vmax, interpolation='nearest')
        ax.set_facecolor(GROUP_SEP[grp])
        pr2 = models_of[tid][key]['pR2']
        if is_top: ax.set_title(f'{title}\n{pr2:+.3f}', fontsize=5.5, color=color, pad=2)
        else:      ax.set_title(f'{pr2:+.3f}', fontsize=5.5, color=color, pad=2)
        ax.set_xticks([]); ax.set_yticks([])

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=c, label=g, edgecolor='#999') for g,c in GROUP_SEP.items()]
fig_of.legend(handles=legend_els, ncol=4, fontsize=7, loc='lower center',
              bbox_to_anchor=(0.5,-0.03), frameon=False)
fig_of.suptitle(f'OF1 — M{mouse} D{day}  |  N_equal={N_equal}', fontsize=10, fontweight='bold')
fig_of.savefig(savepath_of, bbox_inches='tight', dpi=200)
plt.show(); print(f'Saved → {savepath_of}')

## 6b. OF1 — covariate and prediction traces

In [ ]:
TRACE_WINDOW_OF = (0, 2000)  # time-bin range to display — adjust to a good window
TRACE_GAIN_OF   = 5

SHOW_TRACES_OF = [
    ('null',                'Null'),
    ('pos',                 'P'),
    ('pos+spd+hd+hing+lfp', 'P+S+HD+Hing+LFP'),
    ('pos+NGS',             'P+NGS'),
    ('full+NGS',            'Full+NGS'),
]

def draw_covariates_of(ax, px, py, spd, hd, hing, lfp, ngs_traces, window):
    win   = slice(*window)
    t_idx = np.arange(*window)
    h = 1.2
    lpad  = len(t_idx) * 0.30
    wfrac = len(t_idx) * 0.02
    traces = (
        [(px,   'X',     'black',   1.0, 1.3),
         (py,   'Y',     '#555555', 1.0, 1.3),
         (spd,  'Speed', '#9b59b6', 1.0, 1.3),
         (hd,   'HD',    '#e67e22', 1.0, 1.3),
         (hing, 'Hing',  '#16a085', 1.0, 1.3),
         (lfp,  'LFP',   '#27ae60', 1.0, 1.3)] +
        [(t, f'NGS{i+1}', COL_NGS, 0.75, 1.0)
         for i, t in enumerate(ngs_traces)]
    )
    n = len(traces)
    for j, (arr, label, c, a, lw) in enumerate(traces):
        offset = h * (n - 1 - j)
        ax.plot(t_idx, _norm01(arr, win) + offset, lw=lw, alpha=a, color=c)
        ax.text(t_idx[0] - wfrac, offset + h * 0.05,
                label, ha='right', va='bottom', fontsize=5, color=c)
    ax.axhline(h * (n - 6) - h * 0.1, color='lightgray', lw=0.6, ls='--')
    bar_len = int(1000 / time_bs)
    bar_y   = -h * 0.7
    ax.plot([t_idx[0], t_idx[0] + bar_len], [bar_y, bar_y], 'k-', lw=2)
    ax.text(t_idx[0] + bar_len * 0.5, bar_y - h * 0.15,
            '1 s', ha='center', va='top', fontsize=6)
    ax.set_xlim(t_idx[0] - lpad, t_idx[-1] + len(t_idx) * 0.03)
    ax.set_ylim(bar_y - h * 0.4, h * n + 0.6)
    ax.axis('off')

# ── Build figure ─────────────────────────────────────────────────────────────
N_RM_OF  = len(SHOW_MODELS_OF)

col_w_of = [2.0, 0.1, 1.8, 0.2, 2.0, 0.1] + [2.0] * N_RM_OF
n_cols_of_t = len(col_w_of)

fig_oft = plt.figure(figsize=(sum(col_w_of) * 0.75, 3.5))
gs_oft  = gridspec.GridSpec(1, n_cols_of_t, figure=fig_oft,
                             width_ratios=col_w_of, wspace=0.10)

ngs_traces_of = [_pad(np.array(tcs_time_of[nid]), T_of) for nid in top_ngs_ids_of]
draw_covariates_of(fig_oft.add_subplot(gs_oft[0, 0]),
                   px_of, py_of, spd_of, hd_of, hing_of, lfp_of,
                   ngs_traces_of, TRACE_WINDOW_OF)
fig_oft.axes[-1].set_title('Covariates', fontsize=8, pad=4)

fig_oft.add_subplot(gs_oft[0, 1]).axis('off')

draw_predictions(fig_oft.add_subplot(gs_oft[0, 2]),
                 y_of, models_of, SHOW_TRACES_OF, TRACE_WINDOW_OF,
                 TRACE_GAIN_OF, COL_GC)
fig_oft.axes[-1].set_title(f'GC {TARGET_ID} predictions (OF1)', fontsize=8, pad=4)

fig_oft.add_subplot(gs_oft[0, 3]).axis('off')

# true 2D rate map
ax_true_of = fig_oft.add_subplot(gs_oft[0, 4])
ax_true_of.imshow(rm_true_of, origin='lower', cmap='viridis',
                  interpolation='nearest')
ax_true_of.set_title('True', fontsize=8, fontweight='bold')
ax_true_of.set_xticks([]); ax_true_of.set_yticks([])

fig_oft.add_subplot(gs_oft[0, 5]).axis('off')

for pi, key in enumerate(SHOW_MODELS_OF):
    ax = fig_oft.add_subplot(gs_oft[0, pi + 6])
    ax.imshow(rm_preds_of[key], origin='lower', cmap='viridis',
              interpolation='nearest')
    ax.set_title(f'{key}\npR²={models_of[key]["pR2"]:+.3f}', fontsize=6.5, color=COL_GC)
    ax.set_xticks([]); ax.set_yticks([])

fig_oft.suptitle(f'OF1 — M{mouse} D{day}  GC {TARGET_ID}',
                 fontsize=10, fontweight='bold', y=1.02)
fig_oft.savefig(savepath_of, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {savepath_of}')


## 7. Combined VR + OF1 figure

In [ ]:
SHOW_VR = ['pos', 'pos+spd+lfp', 'pos+NGS', 'pos+spd+NGS', 'pos+spd+lfp+NGS']
SHOW_OF = ['pos', 'pos+spd+hd+hing+lfp', 'pos+NGS', 'pos+spd+lfp+NGS', 'full+NGS']
n_pred  = max(len(SHOW_VR), len(SHOW_OF))
n_cols_c = max(n_pred + 1, n_cov)  # +1 for actual

fig_c = plt.figure(figsize=(n_cols_c * 1.8, 9))
gs_outer = gridspec.GridSpec(2, 1, figure=fig_c, hspace=0.25)

def draw_block(gs_sub, session, show_models, rm_true, rm_preds_d,
               top_ids, rm_cov_d, vmax_cov, cmap_cov, cmap_gc,
               is_2d=False):
    gs_inner = gridspec.GridSpecFromSubplotSpec(
        2, n_cols_c, subplot_spec=gs_sub, hspace=0.08, wspace=0.06)

    vmax_gc_c = max(rm_true.max(), max(rm_preds_d[k].max() for k in show_models))

    # Row 0: covariate NGS
    for ci, nid in enumerate(top_ids):
        ax = fig_c.add_subplot(gs_inner[0, ci])
        if is_2d:
            ax.imshow(rm_cov_d[nid], origin='lower', cmap=cmap_cov,
                      vmin=0, vmax=vmax_cov, interpolation='nearest')
        else:
            ax.imshow(rm_cov_d[nid][np.newaxis, :], aspect='auto', cmap=cmap_cov,
                      vmin=0, vmax=vmax_cov, interpolation='nearest')
        ax.set_title(f'NGS {nid}', fontsize=6.5, pad=2)
        ax.set_xticks([]); ax.set_yticks([])
        if ci == 0:
            ax.set_ylabel(f'{session}\nNGS covariates', fontsize=7, rotation=90, labelpad=3)
    for ci in range(len(top_ids), n_cols_c):
        fig_c.add_subplot(gs_inner[0, ci]).axis('off')

    # Row 1: actual + predictions
    ax = fig_c.add_subplot(gs_inner[1, 0])
    if is_2d:
        ax.imshow(rm_true, origin='lower', cmap=cmap_gc,
                  vmin=0, vmax=vmax_gc_c, interpolation='nearest')
    else:
        ax.imshow(rm_true[np.newaxis, :], aspect='auto', cmap=cmap_gc,
                  vmin=0, vmax=vmax_gc_c, interpolation='nearest')
    ax.set_title(f'GC {TARGET_ID}\nActual', fontsize=6.5, pad=2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_ylabel('Target GC\nrate map', fontsize=7, rotation=90, labelpad=3)

    for pi, key in enumerate(show_models):
        ax = fig_c.add_subplot(gs_inner[1, pi + 1])
        rm = rm_preds_d[key]
        if is_2d:
            ax.imshow(rm, origin='lower', cmap=cmap_gc,
                      vmin=0, vmax=vmax_gc_c, interpolation='nearest')
        else:
            ax.imshow(rm[np.newaxis, :], aspect='auto', cmap=cmap_gc,
                      vmin=0, vmax=vmax_gc_c, interpolation='nearest')
        ax.set_title(f'{key}\npR²={rm_preds_d[key]}' if isinstance(rm_preds_d[key], float)
                     else f'{key}\npR²={models_vr[key]["pR2"]:+.3f}'
                     if not is_2d else f'{key}\npR²={models_of[key]["pR2"]:+.3f}',
                     fontsize=6.5, pad=2)
        ax.set_xticks([]); ax.set_yticks([])
    for pi in range(len(show_models) + 1, n_cols_c):
        fig_c.add_subplot(gs_inner[1, pi]).axis('off')


vmax_cov_vr = max(rm_cov_vr[nid].max() for nid in top_ngs_ids)
vmax_cov_of_c = max(rm_cov_of[nid].max() for nid in top_ngs_ids_of)

draw_block(gs_outer[0], 'VR', SHOW_VR, rm_true_vr, rm_preds_vr,
           top_ngs_ids, rm_cov_vr, vmax_cov_vr,
           cmap_cov='Blues', cmap_gc='Reds', is_2d=False)

draw_block(gs_outer[1], 'OF1', SHOW_OF, rm_true_of, rm_preds_of,
           top_ngs_ids_of, rm_cov_of, vmax_cov_of_c,
           cmap_cov='Blues', cmap_gc='Reds', is_2d=True)

fig_c.suptitle(f'M{mouse} D{day}  Target GC: {TARGET_ID}  —  VR (top) and OF1 (bottom)',
               fontsize=10, fontweight='bold')
fig_c.savefig(savepath_combined, bbox_inches='tight', dpi=200)
plt.show()
print(f'Saved → {savepath_combined}')